In [6]:
# Math & Data manipulation
import copy
import json
import numpy as np
import pandas as pd
import scipy as sp
import statsmodels.api as sm
import sympy as sym
import pooch
import os
from pathlib import Path
import gdown

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
#import plotnine

# Single-cell analysis
import scanpy as sc
import anndata as ad

# Machine Learning & Deep Learning
#import scvi
#import sklearn
import torch

from torchtext.vocab import Vocab
from torchtext._torchtext import (
    Vocab as VocabPybind,
)

import scgpt as scg
from scgpt.tasks import GeneEmbedding
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.model import TransformerModel
from scgpt.preprocess import Preprocessor
from scgpt.utils import set_seed

/Users/melinariepl/miniforge3/envs/scgpt_py311/lib/python3.11/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/Users/melinariepl/miniforge3/envs/scgpt_py311/lib/python3.11/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


In [8]:


set_seed(42)
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
n_hvg = 1200
n_bins = 51
mask_value = -1
pad_value = -2
n_input_bins = n_bins



In [9]:


# Target directory
model_dir = "./scGPT_human"
os.makedirs(model_dir, exist_ok=True)

# Direct file IDs extracted from your links
file_ids = {
    "args.json": "1y4UJVflGl-b2qm-fvpxIoQ3XcC2umjj0",
    "best_model.pt": "1MJaavaG0ZZkC_yPO4giGRnuCe3F1zt30",
    "vocab.json": "127FdcUyY1EM7rQfAS0YI4ms6LwjmnT9J",
}

for filename, file_id in file_ids.items():
    output_path = os.path.join(model_dir, filename)

    # Clean up corrupted HTML best_model.pt if it was cached previously
    if filename == "best_model.pt" and os.path.exists(output_path):
        if os.path.getsize(output_path) < 1024 * 1024:  # under 1 MB is just the HTML error
            os.remove(output_path)

    print(f"Downloading {filename}...")

    if filename == "best_model.pt":
        # Google Drive virus warning bypass for large files
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url=url, output=output_path, quiet=False)
    else:
        # Standard pooch retrieval for small metadata files
        url = f"https://drive.google.com/uc?export=download&id={file_id}"
        pooch.retrieve(
            url=url,
            known_hash=None,
            path=model_dir,
            fname=filename,
            progressbar=True
        )

print(f"\nFinished! Files are located in {os.path.abspath(model_dir)}")

Downloading...
From (original): https://drive.google.com/uc?id=1MJaavaG0ZZkC_yPO4giGRnuCe3F1zt30
From (redirected): https://drive.google.com/uc?id=1MJaavaG0ZZkC_yPO4giGRnuCe3F1zt30&confirm=t&uuid=f29d91ab-8bf8-40c2-abb2-f576a4d29318
To: /Users/melinariepl/ramming_lab_code/scGPT_human/best_model.pt
100%|██████████| 156M/156M [03:56<00:00, 659kB/s] 


Finished! Files are located in /Users/melinariepl/ramming_lab_code/scGPT_human


In [15]:
# load adata
adata = sc.read_h5ad("data/adata_integrated.h5ad")


In [16]:

# Specify model path; here we load the pre-trained scGPT blood model
model_dir = Path("./scGPT_human")
model_config_file = model_dir / "args.json"
model_file = model_dir / "best_model.pt"
vocab_file = model_dir / "vocab.json"

vocab = GeneVocab.from_file(vocab_file)
for s in special_tokens:
    if s not in vocab:
        vocab.append_token(s)

# Retrieve model parameters from config files
with open(model_config_file, "r") as f:
    model_configs = json.load(f)
print(
    f"Resume model from {model_file}, the model args will override the "
    f"config {model_config_file}."
)
embsize = model_configs["embsize"]
nhead = model_configs["nheads"]
d_hid = model_configs["d_hid"]
nlayers = model_configs["nlayers"]
n_layers_cls = model_configs["n_layers_cls"]

gene2idx = vocab.get_stoi()



Resume model from scGPT_human/best_model.pt, the model args will override the config scGPT_human/args.json.


In [17]:
# 1. Add Apple Silicon MPS support with fallback to CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

ntokens = len(vocab)  # size of vocabulary
model = TransformerModel(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    vocab=vocab,
    pad_value=pad_value,
    n_input_bins=n_input_bins,
)

try:
    # Always deserialize to CPU first to avoid CUDA device errors
    state_dict = torch.load(model_file, map_location="cpu")
    model.load_state_dict(state_dict)
    print(f"Loading all model params from {model_file}")
except Exception as e:
    print(f"Full load failed ({e}), loading matching params only...")
    model_dict = model.state_dict()
    pretrained_dict = torch.load(model_file, map_location="cpu")
    
    # Filter matching keys and shapes
    pretrained_dict = {
        k: v
        for k, v in pretrained_dict.items()
        if k in model_dict and v.shape == model_dict[k].shape
    }
    
    for k, v in pretrained_dict.items():
        print(f"Loading params {k} with shape {v.shape}")
        
    # Update dict and load once, outside the print loop
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)

# Move the populated model to MPS/CUDA/CPU
model.to(device)

Full load failed (Error(s) in loading state_dict for TransformerModel:
	Missing key(s) in state_dict: "transformer_encoder.layers.0.self_attn.in_proj_weight", "transformer_encoder.layers.0.self_attn.in_proj_bias", "transformer_encoder.layers.1.self_attn.in_proj_weight", "transformer_encoder.layers.1.self_attn.in_proj_bias", "transformer_encoder.layers.2.self_attn.in_proj_weight", "transformer_encoder.layers.2.self_attn.in_proj_bias", "transformer_encoder.layers.3.self_attn.in_proj_weight", "transformer_encoder.layers.3.self_attn.in_proj_bias", "transformer_encoder.layers.4.self_attn.in_proj_weight", "transformer_encoder.layers.4.self_attn.in_proj_bias", "transformer_encoder.layers.5.self_attn.in_proj_weight", "transformer_encoder.layers.5.self_attn.in_proj_bias", "transformer_encoder.layers.6.self_attn.in_proj_weight", "transformer_encoder.layers.6.self_attn.in_proj_bias", "transformer_encoder.layers.7.self_attn.in_proj_weight", "transformer_encoder.layers.7.self_attn.in_proj_bias", "t

TransformerModel(
  (encoder): GeneEncoder(
    (embedding): Embedding(36574, 512, padding_idx=36571)
    (enc_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0.5, inplace=False)
    (linear1): Linear(in_features=1, out_features=512, bias=True)
    (activation): ReLU()
    (linear2): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.5, inplace=False)
        (linear2): Linear(in_features=512, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, el

In [18]:
import sys
import torch

print(f"Python executable: {sys.executable}")
print(f"Python version:    {sys.version.split()[0]}")
print(f"PyTorch version:   {torch.__version__}")

try:
    import numpy as np
    print(f"NumPy version:     {np.__version__}")
    print(f"NumPy path:        {np.__file__}")
except ImportError as e:
    print(f"NumPy import failed: {e}")

# Test PyTorch <-> NumPy bridge
try:
    t = torch.ones(2)
    arr = t.numpy()
    print("PyTorch <-> NumPy bridge: WORKING")
except Exception as e:
    print(f"PyTorch <-> NumPy bridge: FAILED ({type(e).__name__}: {e})")

Python executable: /Users/melinariepl/miniforge3/envs/scgpt_py311/bin/python
Python version:    3.11.16
PyTorch version:   2.2.2
NumPy version:     1.26.4
NumPy path:        /Users/melinariepl/miniforge3/envs/scgpt_py311/lib/python3.11/site-packages/numpy/__init__.py
PyTorch <-> NumPy bridge: WORKING


In [19]:
# Retrieve the data-independent gene embeddings from scGPT
gene_ids = np.array([id for id in gene2idx.values()])
gene_embeddings = model.encoder(torch.tensor(gene_ids, dtype=torch.long).to(device))
gene_embeddings = gene_embeddings.detach().cpu().numpy()

gene_ids

array([36573, 36572, 36571, ...,   682, 32846,  9078])

In [20]:
# Filter on the intersection between the Immune Human HVGs found in step 1.2 and scGPT's 30+K foundation model vocab
gene_embeddings = {gene: gene_embeddings[i] for i, gene in enumerate(gene2idx.keys()) if gene in adata.var.index.tolist()}
print('Retrieved gene embeddings for {} genes.'.format(len(gene_embeddings)))

Retrieved gene embeddings for 1916 genes.


In [21]:
# Construct gene embedding network
embed = GeneEmbedding(gene_embeddings)


100%|██████████| 1916/1916 [00:00<00:00, 1371614.01it/s]
